In [ ]:
adjacency = {
    'A' : ['B', 'G', 'F'],
    'B' : ['A', 'C'],
    'C' : ['B', 'D', 'E', 'F'],
    'D' : ['C'],
    'E' : ['C'],
    'F' : ['A','C','G'],
    'G' : ['A', 'F']
}

colors = ['red', 'green', 'blue', 'yellow']

class DFSAgent:
    def __init__(self, adjacency, colors):
        self.colors = colors
        self.adjacency = adjacency # dizionario di liste
        self.state = {} #{'A':'Red', 'B' : None}

    def valid_coloring(self,state, new_node=None):
        nodes_to_check = state.keys()
        if new_node is not None:
            nodes_to_check = [new_node]

        for node in nodes_to_check:
            for neighbor in self.adjacency[node]:
                if state[node] == state.get(neighbor, None):
                    return False
        return True

    def next_states(self,state):
        for node in self.adjacency:
            if node not in state:
                for color in self.colors:
                    new_state = {**state, node:color}
                    if self.valid_coloring(new_state, new_node=None):
                        yield new_state
                break 

    def dfs(self, state=None):
        if state==None:
            state={}
        if len(state) == len(self.adjacency):
            yield state
        for next_state in self.next_states(state):
            yield from self.dfs(next_state)

            


In [ ]:
# versione ottimizzata 
class DFSAgentOptim(DFSAgent):
    def __init__(self, adjacency, colors, lcv, fwc, mrv):
        super().__init__(adjacency, colors)
        self.lcv, self.fwc, self.mrv = lcv, fwc, mrv

    def legal_values(self, state, node):
        return len(self.colors)-len({state[neighbor] for neighbor in self.adjacency[node] if
                                     neighbor in state})

    def forward_checking(self, state):
        # Simula cosa succede alle altre regioni non ancora assegnate: 
        # se a causa di questo colore anche solo una variabile vicina rimane con 0 colori
        # ammissibili, il controllo fallisce
        return all(self.legal_values(state,node) > 0 for node in 
                   self.adjacency if node not in state) if self.fwc else True

    def MCV(self,state):
        return min ([n for n in self.adjacency if n not in state], key=lambda n:
                     self.legal_values(state, n)) if self.mrv else 0

    def removed_neighbors_colors(self,state,node,color):
        # funzione di utilità che conta quante opzioni rimuove 
        # l'assegnazione di un valore ad una variabile

        neighbors = [n for n in self.adjacency[node] if n not in state]

        return sum(self.legal_values(state, n) for n in neighbors) - sum(
            self.legal_values({**state, node:color}, n) for n in neighbors)


    def LCV_sort(self,state,node):
        # ordiniamo in ordine crescente i colori, serve per selezionare il colore che lascia
        # più libertà all'algoritmo
        return sorted(self.colors, key=lambda c: self.removed_neighbors_colors(state,node,c) if 
                      self.lcv else 0)

    def next_states(self,state):
        node = self.MCV(state)
        return( new_state for color in self.LCV_sort(state, node) if 
               self.valid_coloring(new_state := {**state, node:color}, new_node=node) and
               self.forward_checking(new_state) )

    